# AAROH Text Emotion Transformer

Train, validate, export, reload in a fresh model instance, and verify multilingual GoEmotions/EmoHinD inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_DIR = '/content/AAROH'
if not os.path.isdir(REPO_DIR):
    raise FileNotFoundError(f'Expected repository at {REPO_DIR}')
os.chdir(REPO_DIR)

In [ ]:
%pip install -q torch transformers scikit-learn
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('A CUDA runtime is required for transformer training')

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/AAROH')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'text_emotion'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Path('models/text_emotion').mkdir(parents=True, exist_ok=True)

In [ ]:
!python -m backend.ml.training.train_text_emotion \
  --data-dir datasets/processed \
  --output-dir models/text_emotion \
  --checkpoint-dir checkpoints/text_emotion \
  --drive-checkpoint-dir $CHECKPOINT_DIR \
  --model-name distilbert-base-multilingual-cased \
  --execution-mode PYTORCH_FROZEN \
  --epochs 3 \
  --batch-size 32 \
  --fp16

In [ ]:
from backend.ml.training.models.text_emotion.model import TextEmotionModel
model = TextEmotionModel.load_from_artifact('models/text_emotion', device='cuda')
result = model.encode_and_predict(['Thank you for your help.', 'मुझे बहुत डर लग रहा है।'], device='cuda')
assert len(result['emotion_embeddings']) == 2
assert len(result['emotion_embeddings'][0]) == 768
print('Fresh-process-compatible inference verification passed')

In [ ]:
from pathlib import Path
required = ['config.json','metadata.json','pytorch_model.bin','tokenizer.json','tokenizer_config.json','label_mapping.json','metrics.json']
missing = [name for name in required if not (Path('models/text_emotion') / name).exists()]
if missing: raise FileNotFoundError(missing)
print('Text Emotion artifacts exported:', required)